## Init config

In [ ]:
import torch
from common_functions_python import set_config_file, test_function

import warnings

# Suppress the specific UserWarning
warnings.filterwarnings("ignore", category=UserWarning, message=".*copy constructor.*")
warnings.filterwarnings("ignore", category=UserWarning)

config_file = {
                'name': 'DINO_features',
                'datasets': ['dino_right_large'],
                'bidirectional_lstm': False,
                'lstm_dropout': 0.2,
                'mlp_dropout': 0.2,
                'lr': 0.0002,
                'step_size': 5,
                'gamma': 0.5,
                'weight_decay': 0,
                'hidden_dim': 512,
                'num_layers': 3,
                'batch_size': 32, 
                'frame_frequency': 2,
                'num_epoch': 30,
                'num_workers': 4
                }

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print('device: ', device)
set_config_file(config_file, device)
# test_function()

In [ ]:
import contextlib
import gc
from common_functions_dino_python import train_loop_dino
from common_functions_heatmap_python import train_loop_heatmap

@contextlib.contextmanager
def clear_memory():
    try:
        yield
    finally:
        gc.collect()

datasets_list = [
    ['deephand_left', 'dino_left_large'],
    ['deephand_left', 'dino_left_large', 'dino_right_small'],
    ['deephand_left', 'dino_left_large', 'dino_face_small'],
    ['deephand_left', 'dino_left_large', 'dino_face_small', 'dino_right_small'],
    ['deephand_left', 'dino_left_large', 'heatmap'],
    ['deephand_left', 'dino_left_large', 'dino_face_small', 'heatmap'],
    # ['dino_right_small'],
    # ['deephand_left'],
    # ['dino_left_large'],
    # ['dino_left_small'],
    # ['deephand_left', 'dino_left_large'],
    # ['deephand_left', 'dino_left_large', 'dino_face_small'],
    # ['deephand_left', 'dino_left_large', 'dino_face_small', 'dino_right_small'],
    # ['deephand_left', 'dino_left_small'],
    # ['deephand_left', 'dino_left_small', 'dino_face_small'],
    # ['deephand_left', 'dino_left_small', 'dino_face_small', 'dino_right_small'],
    # ['deephand_left', 'dino_face_small','dino_right_small'],
    # ['dino_left_large', 'dino_face_small', 'dino_right_small'],
    # ['dino_left_small', 'dino_face_small', 'dino_right_small'],
    # ['deephand_left', 'dino_face_small'],
    # ['deephand_left', 'dino_right_small'],
    # ['dino_left_large', 'dino_face_small'],
    # ['dino_left_small', 'dino_face_small'],
    # ['dino_left_large', 'dino_right_small'],
    # ['dino_left_small', 'dino_right_small']
]

dropout_list = [0.1, 0.2]

frame_frequency_list = [2]

bidirectional_lstm_list = [True, False]

for datasets in datasets_list:
    for dropout in dropout_list:
        for frame_frequency in frame_frequency_list:
            for bidirectional_lstm in bidirectional_lstm_list:
                gc.collect()
                with clear_memory():   
                    config_file['datasets'] = datasets
                    config_file['lstm_dropout'] = dropout
                    config_file['mlp_dropout'] = dropout
                    config_file['frame_frequency'] = frame_frequency
                    config_file['bidirectional_lstm'] = bidirectional_lstm
                    set_config_file(config_file, device)

                    if any('heatmap' in s for s in datasets):
                        if not bidirectional_lstm:
                            train_loop_heatmap()
                    else:
                        train_loop_dino()
